# Embeddings — experiment 1: token IDs are addresses

## Goal

See the smallest possible embedding lookup. A token ID is just an integer label; an embedding is a short row of numbers associated with that label.

We will use a tiny pretend vocabulary of five tokens. This is *not* tokenization yet—we choose the IDs ourselves so we can focus on the lookup.

| Token | ID |
| --- | ---: |
| `cat` | 0 |
| `dog` | 1 |
| `runs` | 2 |
| `sleeps` | 3 |
| `.` | 4 |

Each embedding will have 3 numbers, so `embedding_dim = 3`.

In [ ]:
import torch

# Use the Apple GPU (MPS) when this notebook kernel can access it.
# Otherwise use the CPU, so the lesson remains runnable everywhere.
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# A tiny input sequence: cat, sleeps, dog.
# Shape: (3,) because this sequence contains 3 token IDs.
token_ids = torch.tensor([0, 3, 1], device=device)

# One row per possible token ID, and 3 numbers per row.
# Shape: (vocab_size, embedding_dim) = (5, 3).
embedding_table = torch.tensor([
    [0.1, 0.2, 0.3],  # ID 0: cat
    [0.4, 0.5, 0.6],  # ID 1: dog
    [0.7, 0.8, 0.9],  # ID 2: runs
    [1.0, 1.1, 1.2],  # ID 3: sleeps
    [1.3, 1.4, 1.5],  # ID 4: .
], device=device)

# Use each ID as a row address in the table.
# Shape: (3, 3): one 3-number embedding for each of 3 input tokens.
input_embeddings = embedding_table[token_ids]

print('token IDs:', token_ids)
print('embedding table shape:', embedding_table.shape)
print('input embeddings shape:', input_embeddings.shape)
print('input embeddings:\n', input_embeddings)

## What happened?

`token_ids` is `[0, 3, 1]`. Indexing `embedding_table[token_ids]` selects rows 0, 3, and 1—in that exact order. It does not do arithmetic with the IDs: ID `3` simply means “give me row 3.”

The table has shape `(5, 3)`: 5 vocabulary entries, each represented by 3 numbers. The result has shape `(3, 3)`: 3 input tokens, each now represented by 3 numbers. In a trained LLM, these numbers are learned during training; we chose easy-to-read values only for this experiment. The two tensors must live on the same device; here, both are placed on `device`.

## Pause and predict

Before changing or running anything else: if `token_ids` becomes `torch.tensor([4, 0])`, what will be the **shape** of `input_embeddings`, and which two rows of `embedding_table` will it contain? Reply with your prediction.

# Embeddings — experiment 2: PyTorch's embedding layer

## Problem

We can index an ordinary tensor ourselves, but a neural network needs to know which tensors are its learnable parts. `torch.nn.Embedding` is PyTorch's layer for an embedding table.

## Intuition and mathematics

`nn.Embedding(5, 3)` owns a weight table `W` with shape `(5, 3)`. Its lookup rule is exactly the one we used before: for an input ID `t`, the output is row `W[t]`. For a sequence with shape `(3,)`, it produces `(3, 3)`. There is still **no matrix multiplication** here—only row selection.

The layer starts with random values. In a future lesson, training will adjust those values so useful tokens receive useful vectors.

In [4]:
# A seed makes this layer's initial random numbers reproducible.
torch.manual_seed(7)

# 5 possible token IDs (0 through 4); 3 numbers in every embedding.
# The layer owns a learnable weight table with shape (5, 3).
embedding_layer = torch.nn.Embedding(num_embeddings=5, embedding_dim=3, device=device)

# Call the layer with the same IDs as experiment 1.
# Input shape: (3,); output shape: (3, 3).
layer_embeddings = embedding_layer(token_ids)

print('weight table shape:', embedding_layer.weight.shape)
print('input token IDs shape:', token_ids.shape)
print('layer output shape:', layer_embeddings.shape)
print('layer output:\n', layer_embeddings)

weight table shape: torch.Size([5, 3])
input token IDs shape: torch.Size([3])
layer output shape: torch.Size([3, 3])
layer output:
 tensor([[ 0.6883,  0.5535,  0.5519],
        [-0.6302, -0.6239, -0.5062],
        [-0.2440,  0.1579, -2.3352]], device='mps:0',
       grad_fn=<EmbeddingBackward0>)


## What happened?

`embedding_layer.weight` is the table PyTorch will eventually learn. Calling `embedding_layer(token_ids)` selected one row per ID, just as `embedding_table[token_ids]` did in experiment 1. The particular numbers differ because this is a newly initialized random table; its **shape rule** is the important result.

## Pause and predict

Suppose a batch contains two token-ID sequences, each four tokens long. Its input shape is `(2, 4)`. With this layer's `embedding_dim = 3`, what output shape do you predict? Explain what each of the three dimensions represents.

# Embeddings — experiment 3: looking up a batch

## Problem

Models normally process several sequences at once. This collection is called a **batch**. We want the same embedding layer to look up every token in two sequences together.

## Shape rule

Our token IDs have shape `(batch_size, sequence_length) = (2, 4)`. `nn.Embedding` preserves those input dimensions, then appends the embedding dimension: `(2, 4)` becomes `(2, 4, 3)`. In symbols, `output[b, t, :] = W[token_ids[b, t], :]`: at batch item `b` and token position `t`, select one row from the embedding table `W`.

In [3]:
# Two sequences (rows), with four token IDs in each sequence (columns).
# Shape: (2, 4) = (batch_size, sequence_length).
batch_token_ids = torch.tensor([
    [0, 3, 1, 2],  # first sequence
    [4, 0, 0, 1],  # second sequence
], device=device)

# The layer looks up all eight IDs at once.
# Shape: (2, 4, 3) = (batch_size, sequence_length, embedding_dim).
batch_embeddings = embedding_layer(batch_token_ids)

print('batched token IDs shape:', batch_token_ids.shape)
print('batched embeddings shape:', batch_embeddings.shape)
print('second sequence, first token embedding:', batch_embeddings[1, 0])

batched token IDs shape: torch.Size([2, 4])
batched embeddings shape: torch.Size([2, 4, 3])
second sequence, first token embedding: tensor([-2.0510, -0.1138, -0.8796], device='mps:0', grad_fn=<SelectBackward0>)


## What happened?

The output has one vector for every input ID. There are 2 sequences × 4 positions = 8 IDs, and each selected embedding has 3 numbers. `batch_embeddings[1, 0]` means: second sequence (`1` because indexing starts at 0), first token position. Its input ID is `4`, so that vector is row `4` of `embedding_layer.weight`.

## Pause and predict

Without running code: what is the shape of `batch_embeddings[1, 0]`, and which row of `embedding_layer.weight` does it contain?

# Embeddings — experiment 4: prove the lookup rule

## Problem

We have said that an embedding layer selects rows from its weight table. Let’s verify that `embedding_layer(batch_token_ids)` and direct table indexing produce exactly the same tensor.

## Mathematics and shapes

Let `W` be the layer's weight table, with shape `(5, 3)`, and let `T` be our IDs with shape `(2, 4)`. The output `E` has shape `(2, 4, 3)` and follows: `E[b, t, d] = W[T[b, t], d]`. The indices `b`, `t`, and `d` mean batch item, token position, and embedding-number position.

In [5]:
# Directly index the layer's own weight table with the batched token IDs.
# This has shape (2, 4, 3), just like batch_embeddings.
manual_batch_embeddings = embedding_layer.weight[batch_token_ids]

# torch.equal is True only if both tensors have the same shape and values.
same_lookup = torch.equal(batch_embeddings, manual_batch_embeddings)

print('manual lookup shape:', manual_batch_embeddings.shape)
print('layer call equals direct indexing:', same_lookup)

manual lookup shape: torch.Size([2, 4, 3])
layer call equals direct indexing: True


## What happened?

The result is `True`: `nn.Embedding` is a trainable table plus this row-selection operation. PyTorch packages both together so training can find and update `embedding_layer.weight` later.

## Pause and predict

In `batch_token_ids`, ID `0` appears three times. Before running code: do all three output locations receive the same 3-number vector, or three different vectors? Why?

# Tokenization — experiment 1: text to IDs

## Problem

An embedding layer accepts integer IDs, not text such as `"cat sleeps ."`. A **tokenizer** is the component that converts text into tokens and then into their integer IDs.

## Intuition

We will create the smallest kind of tokenizer: a dictionary that maps whole token strings to IDs. First split `"cat sleeps ."` into `["cat", "sleeps", "."]`; then use the dictionary to map those strings to `[0, 3, 4]`. This is deliberately simple. Production LLMs usually use **subword tokenizers**, not space-splitting; we will build up to that later.

## Shapes

Text and the token list are ordinary Python values, so they do not have PyTorch tensor shapes. The final ID tensor has shape `(3,)` because it contains three IDs. Sending it to our embedding layer produces shape `(3, 3)`.

In [ ]:
# A vocabulary maps each known token string to its ID.
vocabulary = {'cat': 0, 'dog': 1, 'runs': 2, 'sleeps': 3, '.': 4}

text = 'cat sleeps .'

# split() separates this deliberately space-separated example into strings.
tokens = text.split()

# For every token string, read its ID from vocabulary, then make one tensor.
# Shape: (3,), because tokens contains three items.
text_token_ids = torch.tensor([vocabulary[token] for token in tokens], device=device)

# The familiar embedding lookup turns the three IDs into three 3-number vectors.
text_embeddings = embedding_layer(text_token_ids)

print('text:', text)
print('tokens:', tokens)
print('token IDs:', text_token_ids)
print('token IDs shape:', text_token_ids.shape)
print('embeddings shape:', text_embeddings.shape)

## What happened?

The dictionary is the vocabulary. Its keys are strings a tokenizer recognizes; its values are the IDs the embedding layer needs. `[vocabulary[token] for token in tokens]` means: go through the token list one item at a time and collect the matching dictionary value. Only after that conversion do we make a PyTorch tensor.

## Pause and predict

Without running code, predict the `tokens` list, the ID tensor values, and its shape for `text = 'dog runs .'`.